# Initialization Case Preprocessing (Standalone)

This notebook demonstrates the preprocessing pipeline used before GaP refinement, without running a simulator:

1. Define a vertical coarse-grid recipe from explicit depths.
2. Generate a `tops_dz.inc` include file.
3. Stage a reusable initialization case from canonical templates in dry-run mode.

It is intentionally separate from the main GaP notebook so preprocessing details do not distract from GaP mesh logic.

In [1]:
from pathlib import Path
import subprocess
import tempfile

from src.WellClass.libs.grid_utils import (
    CoarseGridSpec,
    build_vertical_grid_schedule,
    write_vertical_grid_recipe,
)

In [2]:
spec = CoarseGridSpec(
    top_depth=4.0,
    water_depth=104.0,
    reservoir_top=1004.0,
    bottom_depth=1504.0,
    water_layers=2,
    overburden_layers=16,
    reservoir_layers=80,
)
schedule = build_vertical_grid_schedule(spec)

print(f"Total layers: {len(schedule.dz)}")
print(f"Water layers: {schedule.sections.count('water')}")
print(f"Overburden layers: {schedule.sections.count('overburden')}")
print(f"Reservoir layers: {schedule.sections.count('reservoir')}")
print(f"Depth span: {spec.top_depth} -> {spec.bottom_depth} mMSL")

Total layers: 98
Water layers: 2
Overburden layers: 16
Reservoir layers: 80
Depth span: 4.0 -> 1504.0 mMSL


In [3]:
repo_root = Path.cwd().resolve().parent
preview_file = repo_root / "test_data" / "examples" / "wildcat-pflotran" / "include" / "tops_dz.preview.inc"

write_vertical_grid_recipe(spec, schedule, preview_file, cells_per_layer=400)
print(f"Wrote preview include: {preview_file}")
print()
print(preview_file.read_text(encoding="utf-8"))

Wrote preview include: /workspaces/SCREEN/test_data/examples/wildcat-pflotran/include/tops_dz.preview.inc

EQUALS
TOPS 4 4* 1 1 /
/

DZ
800*50
6400*56.25
32000*6.25
/



In [4]:
repo_root = Path.cwd().resolve().parent

with tempfile.TemporaryDirectory(prefix="screen_init_case_") as tmp:
    output_root = Path(tmp) / "wildcat-init-demo"
    cmd = [
        "python",
        "runscripts/prepare_init_case.py",
        "--output-root", str(output_root),
        "--top-depth", "4",
        "--water-depth", "104",
        "--reservoir-top", "1004",
        "--bottom-depth", "1504",
    ]

    result = subprocess.run(cmd, check=True, capture_output=True, text=True, cwd=repo_root)
    print(result.stdout)

    staged_files = [
        output_root / "model" / "TEMP-0.in",
        output_root / "include" / "TEMP_GRD.grdecl",
        output_root / "include" / "tops_dz.inc",
    ]

    for path in staged_files:
        print(f"{path}: {'OK' if path.exists() else 'MISSING'}")

    print('\nTOPS/DZ from staged case:')
    print((output_root / "include" / "tops_dz.inc").read_text(encoding="utf-8"))

Staged initialization case files:
  deck:   /tmp/screen_init_case_lxnd7mra/wildcat-init-demo/model/TEMP-0.in
  grid:   /tmp/screen_init_case_lxnd7mra/wildcat-init-demo/include/TEMP_GRD.grdecl
  topsdz: /tmp/screen_init_case_lxnd7mra/wildcat-init-demo/include/tops_dz.inc
No --sim-command provided; skipping simulator execution.

/tmp/screen_init_case_lxnd7mra/wildcat-init-demo/model/TEMP-0.in: OK
/tmp/screen_init_case_lxnd7mra/wildcat-init-demo/include/TEMP_GRD.grdecl: OK
/tmp/screen_init_case_lxnd7mra/wildcat-init-demo/include/tops_dz.inc: OK

TOPS/DZ from staged case:
EQUALS
TOPS 4 4* 1 1 /
/

DZ
400*100
3600*100
20000*10
/



## Next Step (Outside This Notebook)

After staging files, run your simulator initialization command (no production objective) to generate `TEMP-0.EGRID` and `TEMP-0.INIT`.

Those generated files are then consumed by the GaP LGR workflow in `03_wellclass_to_gap.ipynb`.